In [1]:
import pandas as pd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

# NEW: handles special characters safely
from urllib.parse import quote_plus  # NEW: handles special characters safely


In [2]:
# Load variables from our .env file into memory
load_dotenv('../.env')

True

opens our .env file and makes its values accessible

In [3]:
# Read the credentials we just stored
host = os.getenv('MYSQL_HOST')
user = os.getenv('MYSQL_USER')
password = quote_plus(os.getenv('MYSQL_PASSWORD'))  # NEW: encode the password
database = os.getenv('MYSQL_DATABASE')

pulls each value out (host, username, password, database name) without ever typing them directly in the code

In [4]:
# Build a connection string and create an "engine" — this is the object
# SQLAlchemy uses to actually talk to MySQL
engine = create_engine(f'mysql+mysqlconnector://{user}:{password}@{host}/{database}')
engine.dispose()  # NEW: clears any old/broken connections before we start

this builds a special connection object. Think of it as "the phone line" between Python and MySQL

---------------------------------------------------------------------------------------------------

**Load CSVs and push them into MySQL**

In [5]:
from sqlalchemy import text

# Map: CSV file path -> MySQL table name
files_to_tables = {
    '../data/raw/olist_customers_dataset.csv': 'customers',
    '../data/raw/olist_geolocation_dataset.csv': 'geolocation',
    '../data/raw/olist_orders_dataset.csv': 'orders',
    '../data/raw/olist_order_items_dataset.csv': 'order_items',
    '../data/raw/olist_order_payments_dataset.csv': 'payments',
    '../data/raw/olist_order_reviews_dataset.csv': 'reviews',
    '../data/raw/olist_products_dataset.csv': 'products',
    '../data/raw/olist_sellers_dataset.csv': 'sellers',
    '../data/raw/product_category_name_translation.csv': 'category_translation'
}

# Some CSVs need their columns renamed to match our MySQL table names exactly
column_renames = {
    'orders': {
        'order_purchase_timestamp': 'purchase_timestamp',
        'order_approved_at': 'approved_at',
        'order_delivered_carrier_date': 'delivered_carrier_date',
        'order_delivered_customer_date': 'delivered_customer_date',
        'order_estimated_delivery_date': 'estimated_delivery_date'
    },
    'products': {
        'product_name_lenght': 'product_name_length',
        'product_description_lenght': 'product_description_length'
    }
}

for file_path, table_name in files_to_tables.items():
    try:
        df = pd.read_csv(file_path)

        if table_name in column_renames:
            df = df.rename(columns=column_renames[table_name])

        with engine.begin() as connection:
            df.to_sql(
                name=table_name,
                con=connection,
                if_exists='append',
                index=False,
                chunksize=5000
            )
        print(f"✅ Loaded {len(df):,} rows into '{table_name}'")
    except Exception as e:
        print(f"❌ FAILED on '{table_name}': {e}")

✅ Loaded 99,441 rows into 'customers'
✅ Loaded 1,000,163 rows into 'geolocation'
✅ Loaded 99,441 rows into 'orders'
✅ Loaded 112,650 rows into 'order_items'
✅ Loaded 103,886 rows into 'payments'
✅ Loaded 99,224 rows into 'reviews'
✅ Loaded 32,951 rows into 'products'
✅ Loaded 3,095 rows into 'sellers'
✅ Loaded 71 rows into 'category_translation'
